00 · Setup
Imports, configurações (tabelas, pastas, paleta) e funções usadas por todos os notebooks.
Não roda sozinho: é chamado com `%run ./00_setup`.

In [0]:
#%pip install seaborn -q

Imports

In [0]:


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from matplotlib.colors import LinearSegmentedColormap


Configurações

In [0]:
# ---------- Datas do projeto ----------
DT_INICIO        = '2024-03-01'   # 1ª safra: 2 meses de histórico antes
DT_OOT           = '2026-06-01'   # safra out-of-time, guardada para validar o modelo
DT_INICIO_CURSOS = '2025-03-01'   # 1ª safra com histórico da plataforma de cursos
DT_SCORE  = '2026-09-01'   # safra a pontuar (sem rótulo: dados só até 07/09)

VERSAO = 'v0'    
# ---------- Tabelas de origem ----------
SCHEMA = 'workspace.tmw_loyalty'
TABELA_ABT       = f'{SCHEMA}.abt_churn'
T_CLIENTES          = f'{SCHEMA}.clientes'
T_TRANSACOES        = f'{SCHEMA}.transacoes'
T_TRANSACAO_PRODUTO = f'{SCHEMA}.transacao_produto'
T_PRODUTOS          = f'{SCHEMA}.produtos'

SCHEMA_EDU               = 'workspace.tmw_education'
T_EDU_CURSOS             = f'{SCHEMA_EDU}.cursos'
T_EDU_EPS                = f'{SCHEMA_EDU}.cursos_episodios'
T_EDU_EPS_COMPLETOS      = f'{SCHEMA_EDU}.cursos_episodios_completos'
T_EDU_USUARIOS           = f'{SCHEMA_EDU}.usuarios_tmw'
T_EDU_HABILIDADES        = f'{SCHEMA_EDU}.habilidades'
T_EDU_HABILIDADES_CARGOS = f'{SCHEMA_EDU}.habilidades_cargos'
T_EDU_HABILIDADES_USU    = f'{SCHEMA_EDU}.habilidades_usuarios'





In [0]:
MESES = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']

def mes(dt, ano4=False):
    """'2026-06-01' -> 'jun/26' (ou 'jun/2026' com ano4=True)."""
    dt = pd.Timestamp(dt)
    ano = dt.year if ano4 else f'{dt.year % 100:02d}'
    return f'{MESES[dt.month - 1]}/{ano}'

DT_FIM_TREINO  = (pd.Timestamp(DT_OOT) - pd.DateOffset(months=1)).strftime('%Y-%m-%d')   # última safra de treino
PERIODO_TREINO = f'{mes(DT_INICIO)} a {mes(DT_FIM_TREINO)}'                           # ex.: 'mar/24 a mai/26'
_pos_oot = pd.date_range(pd.Timestamp(DT_OOT) + pd.DateOffset(months=1),
                         pd.Timestamp(DT_SCORE) - pd.DateOffset(months=1), freq='MS')
PERIODO_POS_OOT = f'{mes(_pos_oot[0])} a {mes(_pos_oot[-1])}' if len(_pos_oot) else 'nenhuma safra'

In [0]:
# ---------- Pastas de saída ----------
USUARIO      = spark.sql('SELECT current_user()').first()[0]    # e-mail
PROJECT_ROOT = f'/Workspace/Users/{USUARIO}/ds-churn-teo-me-why'
FIG_DIR      = f'{PROJECT_ROOT}/output/figs'
TABLE_DIR    = f'{PROJECT_ROOT}/output/tables'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
README_DIR = f'{FIG_DIR}/readme'
os.makedirs(README_DIR, exist_ok=True)
FONTE_README = f'Fonte: sistema de pontos Téo Me Why · usuários ativos, safras mensais de {PERIODO_TREINO}'

In [0]:
# ---------- Paleta da marca (estrutura e gráficos sem sinal de churn) ----------
COLOR_DARK  = '#3c2f2f'   # textos, eixos, pontos
COLOR_BLUE  = '#00C2F0'   # azul da marca: barras e linhas neutras
COLOR_TITLE = '#003d48'   # títulos e cabeçalhos
BLUE_PALETTE = ['#003d48', '#006d80', '#00C2F0', '#7fdff7']   # séries múltiplas (escuro -> claro)

# ---------- Cores de sinal (só quando a cor indica churn / ativo / referência) ----------
COR_NEUTRA = '#6C757D'   # linha de média / referência

# mapa de cor sequencial da marca, para heatmaps de volume (não de churn)
CMAP_MARCA = LinearSegmentedColormap.from_list('marca', ['#ffffff', COLOR_BLUE, COLOR_TITLE])

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlecolor': COLOR_TITLE,
    'figure.titlesize': 14, 'figure.titleweight': 'bold',
    'axes.labelsize': 10, 'axes.labelcolor': COLOR_DARK,
    'xtick.color': COLOR_DARK, 'ytick.color': COLOR_DARK,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'text.color': COLOR_DARK, 'axes.edgecolor': COLOR_DARK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.4,
    'legend.fontsize': 9, 'legend.frameon': False,
})

### Funções

In [0]:
def salvar(fig, arquivo):
    plt.tight_layout()
    fig.savefig(f'{FIG_DIR}/{arquivo}.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


def univariada(serie, titulo, arquivo, xlabel=None, bins=30, discreta=False):
    """Como a variável é: resumo estatístico + histograma (ou barras, se discreta)."""
    s = serie.dropna()
    if pd.api.types.is_numeric_dtype(s):
        print(s.describe().round(2).to_string())
        print(f'% de zeros: {(s == 0).mean():.1%}   |   % missing: {serie.isna().mean():.1%}')
    else:
        print(s.value_counts(normalize=True).round(3).to_string())
    fig, ax = plt.subplots()
    if discreta:
        s.value_counts().sort_index().plot.bar(ax=ax, color=COLOR_BLUE)
    else:
        s.plot.hist(bins=bins, ax=ax, color=COLOR_BLUE, edgecolor='white')
    ax.set(title=titulo, xlabel=xlabel or serie.name, ylabel='nº de usuários')
    salvar(fig, arquivo)


def _barras_churn(res, taxa_media, titulo, xlabel, arquivo, destacar=True):
    fig, ax = plt.subplots()
    rotulos = [f'{i}\n(n={n:,})'.replace(',', '.') for i, n in zip(res.index.astype(str), res['n'])]

    # azul da marca; se destacar=True, faixas acima da média ficam no azul escuro
    if destacar:
        cores = [COLOR_TITLE if t > taxa_media else COLOR_BLUE for t in res['taxa']]
    else:
        cores = COLOR_BLUE

    ax.bar(rotulos, res['taxa'] * 100, yerr=res['ic'] * 100,
           color=cores, ecolor=COLOR_DARK, capsize=4)
    ax.axhline(taxa_media * 100, ls='--', color=COR_NEUTRA, label=f'média geral ({taxa_media:.0%})')
    ax.set(title=titulo, xlabel=xlabel, ylabel='taxa de churn (%)')
    ax.legend()
    salvar(fig, arquivo)


def _resumo(df, grupo):
    res = df.groupby(grupo, observed=True)['churn'].agg(taxa='mean', n='size')
    res['ic'] = 1.96 * np.sqrt(res['taxa'] * (1 - res['taxa']) / res['n'])
    return res


def bivariada(df, col, bins, labels, titulo, arquivo, xlabel=None):
    """Taxa de churn por faixa de uma variável numérica."""
    faixa = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True, right=True)
    res = _resumo(df.assign(_faixa=faixa), '_faixa')
    _barras_churn(res, df['churn'].mean(), titulo, xlabel or col, arquivo)
    return res.drop(columns='ic').round(3)


def bivariada_cat(df, col, titulo, arquivo, xlabel=None, ordem=None):
    """Taxa de churn por categoria."""
    res = _resumo(df, col)
    if ordem is not None:
        res = res.reindex([o for o in ordem if o in res.index])
    else:
        res = res.sort_values('taxa', ascending=False)
    _barras_churn(res, df['churn'].mean(), titulo, xlabel or col, arquivo)
    return res.drop(columns='ic').round(3)


def multivariada(df, col_x, col_y, titulo, arquivo, n_min=30):
    """Heatmap: taxa de churn (%) em cada combinação de duas variáveis em faixas.
    Células com menos de n_min pessoas ficam em branco."""
    taxa = df.pivot_table(index=col_y, columns=col_x, values='churn', aggfunc='mean', observed=True) * 100
    n = df.pivot_table(index=col_y, columns=col_x, values='churn', aggfunc='size', observed=True)
    taxa = taxa.where(n >= n_min)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(taxa, annot=True, fmt='.0f', cmap=CMAP_MARCA, ax=ax,     
                cbar_kws={'label': 'taxa de churn (%)'})
    ax.grid(False)
    ax.set(title=titulo, xlabel=col_x, ylabel=col_y)
    salvar(fig, arquivo)
    return taxa.round(1)


def perfil_tabela(tabela, chave):
    """Raio-x de uma tabela: tamanho, tipos, nulos, distintos, unicidade da chave, describe e datas.
    chave: nome de uma coluna ou lista de colunas (chave composta)."""
    sdf = spark.table(tabela)
    chaves = [chave] if isinstance(chave, str) else list(chave)
    n_linhas = sdf.count()
    n_chave = sdf.select(*chaves).distinct().count()
    nome_chave = ' + '.join(chaves)
    print(f'Tabela: {tabela}')
    print(f'Linhas: {n_linhas:,}   |   Colunas: {len(sdf.columns)}'.replace(',', '.'))
    print(f'Chave {nome_chave}: {n_chave:,} combinações distintas -> '.replace(',', '.') +
          ('única ✅' if n_chave == n_linhas else f'NÃO é única ⚠️ ({n_linhas - n_chave:,} repetidas)'.replace(',', '.')))
    print(f'Linhas totalmente duplicadas: {n_linhas - sdf.distinct().count():,}'.replace(',', '.'))

    # nulos e distintos por coluna
    agg = sdf.agg(
        *[F.sum(F.col(c).isNull().cast('int')).alias(f'{c}__nulos') for c in sdf.columns],
        *[F.countDistinct(F.col(c)).alias(f'{c}__distintos') for c in sdf.columns],
    ).first().asDict()
    resumo = pd.DataFrame({
        'tipo': [t for _, t in sdf.dtypes],
        'nulos': [agg[f'{c}__nulos'] for c in sdf.columns],
        'distintos': [agg[f'{c}__distintos'] for c in sdf.columns],
    }, index=sdf.columns)
    resumo['% nulos'] = (resumo['nulos'] / n_linhas * 100).round(2)
    resumo['constante?'] = np.where(resumo['distintos'] <= 1, 'sim ⚠️', '')
    display(resumo)

    # describe das numéricas
    numericas = [c for c, t in sdf.dtypes if t in ('int', 'bigint', 'double', 'float', 'smallint', 'tinyint')
                 or t.startswith('decimal')]
    if numericas:
        print('Describe das colunas numéricas:')
        display(sdf.select(numericas).summary('count', 'mean', 'stddev', 'min', '25%', '50%', '75%', 'max')
                   .toPandas().set_index('summary'))

    # período coberto pelas datas
    datas = [c for c, t in sdf.dtypes if t in ('timestamp', 'date')]
    for c in datas:
        mn, mx = sdf.agg(F.min(c), F.max(c)).first()
        print(f'{c}: de {mn} até {mx}')

    print('Amostra:')
    display(sdf.limit(5).toPandas())
    return resumo

def carregar_abt(base='treino'):
    """base: 'treino' (< OOT), 'oot' (= OOT) ou 'score' (= DT_SCORE)."""
    df = spark.table(TABELA_ABT).filter(f"baseSafra = '{base}'").toPandas()
    df['tendencia'] = pd.Categorical(df['tendencia'], ['caindo', 'estável', 'crescendo'], ordered=True)
    return df


def painel_churn(df, specs, titulo, arquivo):
    """Vários gráficos de taxa de churn lado a lado. specs = [(coluna, ordem das categorias, subtítulo), ...]"""
    media = df['churn'].mean()
    fig, axes = plt.subplots(1, len(specs), figsize=(5 * len(specs), 4.5), sharey=True)
    for ax, (col, ordem, subtitulo) in zip(axes, specs):
        res = _resumo(df, col).reindex(ordem)
        rotulos = [f'{i}\n(n={n:,})'.replace(',', '.') for i, n in zip(res.index, res['n'])]
        cores = [COLOR_TITLE if t > media else COLOR_BLUE for t in res['taxa']]
        ax.bar(rotulos, res['taxa'] * 100, yerr=res['ic'] * 100, color=cores, ecolor=COLOR_DARK, capsize=4)
        ax.axhline(media * 100, ls='--', color=COR_NEUTRA)
        ax.set(title=subtitulo, xlabel='')
    axes[0].set_ylabel('taxa de churn (%)')
    fig.suptitle(titulo, fontweight='bold', color=COLOR_TITLE)
    salvar(fig, arquivo)

def moldura(fig, ax, titulo, subtitulo, fonte=FONTE_README):
    """Padrão das figuras do README: título (mensagem), subtítulo (contexto) e fonte no rodapé."""
    ax.set_title('')
    fig.suptitle(titulo, x=0.01, ha='left', fontsize=15, fontweight='bold', color=COLOR_TITLE)
    fig.text(0.01, 0.905, subtitulo, ha='left', fontsize=10, color=COLOR_DARK)
    fig.text(0.01, -0.06, fonte, ha='left', fontsize=8, color=COR_NEUTRA)
    fig.subplots_adjust(top=0.82)


def salvar_readme(fig, arquivo):
    fig.savefig(f'{README_DIR}/{arquivo}.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


def pct(x):
    return f'{x:.0%}'



In [0]:
# ---------- Textos montados com f-string ----------
def pct1(x):
    """0.479 -> '47,9%'"""
    return f'{x:.1%}'.replace('.', ',')


def dec(x, casas=3):
    """0.8061 -> '0,806'"""
    return f'{x:.{casas}f}'.replace('.', ',')


def num(x):
    """10554 -> '10.554'"""
    return f'{x:,.0f}'.replace(',', '.')


def md(texto):
    """Mostra um texto em Markdown. Use com f-string para que datas e números venham do código."""
    from IPython.display import Markdown, display as ipy_display
    ipy_display(Markdown(texto))


NOME_MODELO = {
    'arvore_decisao': 'Árvore de decisão', 'logistica': 'Regressão logística', 'random_forest': 'Random Forest',
    'adaboost': 'AdaBoost', 'gradient_boosting': 'Gradient Boosting', 'hist_gradient_boosting': 'HistGradientBoosting',
    'xgboost': 'XGBoost', 'lightgbm': 'LightGBM', 'catboost': 'CatBoost', 'mlp': 'MLP', 'svm': 'SVM',
}

# números-chave de cada notebook, guardados num JSON: o 07_readme lê daqui para montar o README
NUMEROS_JSON = f'{TABLE_DIR}/numeros_{VERSAO}.json'


def registrar(**valores):
    """Guarda números-chave no JSON (junta com o que já existe)."""
    import json
    try:
        with open(NUMEROS_JSON) as f:
            atual = json.load(f)
    except FileNotFoundError:
        atual = {}
    for chave, v in valores.items():
        if isinstance(v, np.generic):
            v = v.item()
        elif isinstance(v, pd.Timestamp):
            v = v.strftime('%Y-%m-%d')
        atual[chave] = v
    with open(NUMEROS_JSON, 'w') as f:
        json.dump(atual, f, ensure_ascii=False, indent=2, sort_keys=True)


def numeros():
    """Lê os números-chave guardados por registrar()."""
    import json
    with open(NUMEROS_JSON) as f:
        return json.load(f)